In [2]:
import torch
import torch.nn as nn

In [ ]:
#For text models
class RMSNorm(nn.Module):
    def __init__(self,dim, eps = 1e-6):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim,dtype=torch.float32),requires_grad=True)
        self.eps = eps
    def forward(self, x: torch.Tensor):
        _,_,d = x.shape
        y = x
        x = x**2
        den = torch.sqrt(
            self.eps + (x.sum(dim=-1,keepdim=True)/d)
        )
        y = y / den
        return self.scale * y 

In [4]:
se = RMSNorm([4])
t = nn.RMSNorm([4])
x = torch.randn(2,2,4)
se(x),t(x)

(tensor([[[ 0.0107,  1.5217,  0.1732, -1.2862],
          [-0.9180,  0.5926,  1.1011,  1.2625]],
 
         [[-0.5634, -1.0139, -1.2821,  1.0055],
          [ 0.5056, -1.5660,  0.3939,  1.0662]]], grad_fn=<MulBackward0>),
 tensor([[[ 0.0107,  1.5217,  0.1732, -1.2862],
          [-0.9180,  0.5926,  1.1011,  1.2625]],
 
         [[-0.5634, -1.0139, -1.2821,  1.0055],
          [ 0.5056, -1.5660,  0.3939,  1.0663]]], grad_fn=<MulBackward0>))

#### ref: https://nn.labml.ai/transformers/rope/index.html

In [ ]:
class RoPE(nn.Module):
    def __init__(self, d: int, base: int = 10_000):
        super().__init__()
        self.base = base
        self.d = d
        self.cos_cached = None
        self.sin_cached = None
        
    def _build_cache(self, seq_len: int, device):
        if self.cos_cached is not None and seq_len <= self.cos_cached.shape[0]:
            if self.cos_cached.device == device:
                return
        
        theta = 1 / (self.base ** (torch.arange(0, self.d, 2, device=device).float() / self.d))
        seq_idx = torch.arange(seq_len, device=device).float()
        idx_theta = seq_idx[:, None] * theta
        idx_theta2 = torch.cat([idx_theta, idx_theta], dim=1)
        
        self.cos_cached = idx_theta2.cos()[None, :, None, :]  # (1, T, 1, D)
        self.sin_cached = idx_theta2.sin()[None, :, None, :]  # (1, T, 1, D)
    
    def _neg_half(self, x_rope):
        d_by_2 = self.d // 2
        return torch.cat([-x_rope[..., d_by_2:], x_rope[..., :d_by_2]], dim=-1)
    
    def forward(self, x, offset: int = 0):
        """
        x: (B, T, H, D)
        offset: position offset for KV caching
        """
        seq_len = x.shape[1] + offset
        self._build_cache(seq_len, x.device)
        
        T = x.shape[1]
        x_rope, x_pass = x[..., :self.d], x[..., self.d:]
        neg_half_x = self._neg_half(x_rope)
        
        # Apply rotation for positions [offset:offset+T]
        cos = self.cos_cached[:, offset:offset+T]  # (1, T, 1, D)
        sin = self.sin_cached[:, offset:offset+T]  # (1, T, 1, D)
        
        x_rope = (x_rope * cos) + (neg_half_x * sin)
        return torch.cat([x_rope, x_pass], dim=-1)

In [82]:
d=8
base=10_000
seq_len = 10
rope = RoPE(4)
inp = torch.randn(seq_len,2,2,d)
rope(inp)[2] , inp[2]

(tensor([[[-0.6584, -0.4778, -0.3544,  0.1811, -0.4183, -0.1948,  1.1654,
            2.4231],
          [ 0.7754,  0.5845, -0.6305, -1.9624, -2.5552, -1.5571,  0.1501,
            0.7989]],
 
         [[ 0.3926,  0.9148, -0.1711,  0.1695, -0.7624, -0.7864,  0.6547,
            0.7747],
          [-0.1333,  0.2730, -0.5014, -0.8523,  1.9975,  1.1279, -0.5007,
            1.1493]]]),
 tensor([[[-0.0483, -0.4741,  0.7461,  0.1906, -0.4183, -0.1948,  1.1654,
            2.4231],
          [-0.8960,  0.5451, -0.4427, -1.9737, -2.5552, -1.5571,  0.1501,
            0.7989]],
 
         [[-0.3189,  0.9180, -0.2858,  0.1511, -0.7624, -0.7864,  0.6547,
            0.7747],
          [-0.4004,  0.2559,  0.3298, -0.8576,  1.9975,  1.1279, -0.5007,
            1.1493]]]))

<!-- # Initialize
rope = RoPE(d=128)
attn = GQFlashAttention(d_model=512, num_q_heads=8, num_kv_heads=2, rope=rope)

# Prefill phase
kv_cache = {}
out = attn._prefill(prompt_tokens, kv_cache)

# Decode phase (autoregressive generation)
for _ in range(num_new_tokens):
    out, kv_cache = attn.decode(next_token, kv_cache) -->


#### Initialize
```python
rope = RoPE(d=128)
attn = GQFlashAttention(d_model=512, num_q_heads=8, num_kv_heads=2, rope=rope)
```

#### Prefill phase
```python
kv_cache = {}
out = attn._prefill(prompt_tokens, kv_cache)
```

#### Decode phase (autoregressive generation)
```python
for _ in range(num_new_tokens):
    out, kv_cache = attn.decode(next_token, kv_cache)
```

<!-- # Initialize
rope = RoPE(d=128)
attn = GQFlashAttention(d_model=512, num_q_heads=8, num_kv_heads=2, rope=rope)

# Prefill phase
kv_cache = {}
out = attn._prefill(prompt_tokens, kv_cache)

# Decode phase (autoregressive generation)
for _ in range(num_new_tokens):
    out, kv_cache = attn.decode(next_token, kv_cache) -->


In [ ]:
import torch.nn.functional as F
try:
    from flash_attn.flash_attn_interface import flash_attn_func
    FLASH_AVAILABLE = True
except Exception:
    FLASH_AVAILABLE = False
class GQFlashAttention(nn.Module):
    def __init__(
            self,
            d_model: int,
            num_q_heads: int,
            num_kv_heads: int,
            rope: nn.Module
    ):
        super().__init__()

        assert num_q_heads % num_kv_heads ==0
        self.d_model = d_model
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_q_heads
        self.group_size = num_q_heads // num_kv_heads

        #projections 
        self.w_q = nn.Linear(d_model, num_q_heads * self.head_dim, bias=False)
        self.w_k = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.w_v = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.O = nn.Linear(d_model,d_model,bias=False)

        self.rope = rope
    def _expand_kv(self,k,v):
        if self.group_size==1:
            return k,v
        k = k.repeat_interleave(self.group_size,dim=1)
        v = v.repeat_interleave(self.group_size,dim=1)

        return k,v
    def forward(self,x: torch.Tensor):
        B, T, _ = x.shape

        #project
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        #Reshape into head dim
        q = q.view(B,T,self.num_q_heads,self.head_dim)
        k = k.view(B,T,self.num_kv_heads,self.head_dim)
        v = v.view(B,T,self.num_kv_heads,self.head_dim)

        q = self.rope(q)
        k = self.rope(k)

        out = self._attention(q,k,v,is_causal=True)
        out = out.reshape(B, T, self.d_model)
        return self.O(out)

    def _attention(self,q,k,v,is_causal: bool):
        """
        q: (B, Tq, Hq,  Dh)
        k: (B, Tk, Hkv, Dh)
        v: (B, Tk, Hkv, Dh)
        """
        if FLASH_AVAILABLE and q.is_cuda:
            return flash_attn_func(
                q,k,v,causal=is_causal
            )
        k,v = self._expand_kv(
            k.transpose(1,2),v.transpose(1,2)
        )
        q = q.transpose(1,2)
        out = F.scaled_dot_product_attention(
            q,k,v,
            attn_mask=None,
            dropout_p=0.,
            is_causal=is_causal
        )
        return out.transpose(1,2)
    def _prefill(self,x,kv_cache):
        """
        x: (B, T, d_model)
        T -> prompt_len
        kv_cache: dict with keys 'k', 'v'
        """
        B, T, _ = x.shape
        q = self.w_q(x).view(B,T,self.num_q_heads,self.head_dim)
        k = self.w_k(x).view(B,T,self.num_kv_heads,self.head_dim)
        v = self.w_v(x).view(B,T,self.num_kv_heads,self.head_dim)

        q = self.rope(q)
        k = self.rope(k)

        kv_cache["k"] = k
        kv_cache['v'] = v

        out = self._attention(q,k,v,is_causal=True)
        out = out.reshape(B,T,self.d_model)

        return self.O(out)

    def decode(self,x,kv_cache):
        """
        x: (B, 1, d_model)
        kv_cache: dict with keys 'k', 'v'
        """
        B, _, _ = x.shape
        q = self.w_q(x).view(B,1,self.num_q_heads,self.head_dim)
        k = self.w_k(x).view(B,1,self.num_kv_heads,self.head_dim)
        v = self.w_v(x).view(B,1,self.num_kv_heads,self.head_dim)

        pos = kv_cache['k'].shape[1]
        q = self.rope(q,offset = pos)
        k = self.rope(k,offset = pos)

        kv_cache['k'] = torch.cat([kv_cache['k'],k],dim=1)
        kv_cache['v'] = torch.cat([kv_cache['v'],v],dim=1)

        out = self._attention(
            q,kv_cache['k'],kv_cache['v'],is_causal=False
        )
        out = out.reshape(B,1,self.d_model)
        return self.O(out), kv_cache
    